In [1]:
import json
import pandas as pd

from Pipeline.Model.GallstoneDataSet import GallstoneDataSet
from Pipeline.Model.ExtremeLearningMachine import ExtremeLearningMachine, sigmoid
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix

In [2]:
config_file = '../../Dataset/JSON/full_model_configs.json'

model_configs = []

try:
    with open(config_file, 'r') as f:
        configs = json.load(f)
        for config in configs:
            if config["Activation"] == "sigmoid":
                config["Activation"] = sigmoid
        model_configs.extend(configs)
except FileNotFoundError:
    print(f"Error: {config_file} not found. Run ELM_Evaluation and Grid_Evaluation first.")

print(f"Loaded {len(model_configs)} configurations ready for testing.")

Loaded 3 configurations ready for testing.


In [3]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()
features_size = gallstone_dataset.x_train_scaled.shape[1]

In [4]:
x_test_scaled = gallstone_dataset.x_test_scaled
y_test = gallstone_dataset.y_test
x_train_scaled = gallstone_dataset.x_train_scaled
y_train = gallstone_dataset.y_train

In [5]:
testing_results = []
for config in model_configs:
    print(f"Executing: Nodes : {config['Hidden_Nodes']} , lambda value : {config['Lambda_Value']} , activation function : {config['Activation'].__name__}")

    elm = ExtremeLearningMachine(
        features_size   = features_size,
        hidden_size     = config["Hidden_Nodes"],
        activation_function     = config["Activation"],
        regularization_lambda   = config["Lambda_Value"]
    )
    elm.initialize_random_weights(random_seed=42)


    elm.fit(x_train_scaled.values, y_train.values)
    y_pred = elm.predict(x_test_scaled.values)

    evaluation = EvaluationMatrix(y_test, y_pred)
    print(evaluation.get_report())

    metrics = evaluation.get_all_metrics()
    test_result = {
        "Model_Type"   : config.get('Model_Types', 'Unknown'), # Extract the strategy name
        "Hidden_Nodes" : config['Hidden_Nodes'],
        "Lambda_Value" : config['Lambda_Value'],
        "Activation"   : config['Activation'].__name__
    }
    test_result.update(metrics)
    testing_results.append(test_result)
    print("\n")

Executing: Nodes : 58 , lambda value : 0.0 , activation function : sigmoid
{'Counts': {'TP': np.int64(22), 'TN': np.int64(24), 'FP': np.int64(8), 'FN': np.int64(10)}, 'Metrics': {'Accuracy': np.float64(0.7188), 'Precision': 0.7333, 'Recall': 0.6875, 'NPV': 0.7059, 'Specificity': 0.75, 'F1-Score': 0.7097, 'F2-Score': 0.6962, 'Bal Accuracy': 0.7188, 'MCC': 0.4384}}


Executing: Nodes : 58 , lambda value : 0.000244140625 , activation function : sigmoid
{'Counts': {'TP': np.int64(22), 'TN': np.int64(24), 'FP': np.int64(8), 'FN': np.int64(10)}, 'Metrics': {'Accuracy': np.float64(0.7188), 'Precision': 0.7333, 'Recall': 0.6875, 'NPV': 0.7059, 'Specificity': 0.75, 'F1-Score': 0.7097, 'F2-Score': 0.6962, 'Bal Accuracy': 0.7188, 'MCC': 0.4384}}


Executing: Nodes : 58 , lambda value : 0.000244140625 , activation function : sigmoid
{'Counts': {'TP': np.int64(22), 'TN': np.int64(24), 'FP': np.int64(8), 'FN': np.int64(10)}, 'Metrics': {'Accuracy': np.float64(0.7188), 'Precision': 0.7333, 'Recall': 

In [6]:
final_model_result = pd.DataFrame(testing_results)
final_model_result

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy,Precision,Recall,NPV,Specificity,F1-Score,F2-Score,Bal Accuracy,MCC
0,Best_Hidden_Nodes,58,0.000000,sigmoid,0.71875,0.733333,0.6875,0.705882,0.75,0.709677,0.696203,0.71875,0.438357
1,Best_Lambda,58,0.000244,sigmoid,0.71875,0.733333,0.6875,0.705882,0.75,0.709677,0.696203,0.71875,0.438357
2,Best_Size_and_Lambda,58,0.000244,sigmoid,0.71875,0.733333,0.6875,0.705882,0.75,0.709677,0.696203,0.71875,0.438357
